# Analyze Orphanet using ENRICHR

In [1]:
import argparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
import os
import requests
import json
import pickle
import time
from common_utils import *
from hidef_multiplex_adapter import MultiplexClusterLoaderMixin
import importlib, pipeline_config, multiple_testing
importlib.reload(pipeline_config); importlib.reload(multiple_testing)
from multiple_testing import adjust_pvalues, get_alpha, method_label

In [2]:
class OrphanetDiseaseEnrichrAnalyzer(MultiplexClusterLoaderMixin):
    def __init__(self, ppi_source, input_dir, output_dir, persistence_threshold):
        import pipeline_config as cfg
        if persistence_threshold is None:
            persistence_threshold = cfg.PERSISTENCE_THRESHOLD
        print(f"\n{'='*80}")
        print(f"INITIALIZING OrphanetDiseaseEnrichrAnalyzer")
        print(f"{'='*80}")
        print(f"PPI Source: {ppi_source}")
        print(f"Input Dir: {input_dir}")
        print(f"Output Dir: {output_dir}")
        print(f"Multiple testing: {method_label()} (alpha={get_alpha()})")
        
        self.ppi_source = ppi_source
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.cluster_type = 'hidef_from_comparison'
        self.persistence_threshold = persistence_threshold

        # Enrichr API endpoints
        self.enrichr_url = 'https://maayanlab.cloud/Enrichr'
        self.add_list_url = f'{self.enrichr_url}/addList'
        self.enrich_url = f'{self.enrichr_url}/enrich'
        
        # Orphanet library name in Enrichr
        self.library_name = 'Orphanet_Augmented_2021'
        
        print(f"\nEnrichr Configuration:")
        print(f"  Base URL: {self.enrichr_url}")
        print(f"  Library: {self.library_name}")
        
        self.analysis_dir = os.path.join(output_dir, ppi_source, f'orphanet_enrichr_analysis_{self.cluster_type}')
           
        print(f"\nOutput Configuration:")
        print(f"  Analysis directory: {self.analysis_dir}")
        
        ensure_dir(self.analysis_dir)
        print(f"  Directory created/verified: {os.path.exists(self.analysis_dir)}")
        
        self.clusters = None
        self.seed_genes = None
        self.cluster_seed_mapping = None
        self.results = []
        self.gene_results = []
        
        print(f"{'='*80}\n")

    @property
    def safe_prefix(self):
        """Return a safe prefix for filename generation"""
        return self.cluster_type
    
    def safe_log2(self, value, min_value=1e-10, max_value=1e10):
        """Safely calculate log2 with handling for zero/negative values and infinite results."""
        bounded_value = max(min_value, min(value, max_value))
        return np.log2(bounded_value)
        
    def submit_gene_list_to_enrichr(self, gene_list, description=""):
        """Submit a gene list to Enrichr and return the user list ID."""
        print(f"    Submitting {len(gene_list)} genes to Enrichr...")
        print(f"    First 5 genes: {gene_list[:5]}")
        
        genes_str = '\n'.join(gene_list)
        payload = {
            'list': (None, genes_str),
            'description': (None, description)
        }
        
        try:
            response = requests.post(self.add_list_url, files=payload)
            print(f"    Response status code: {response.status_code}")
            
            if response.ok:
                data = response.json()
                user_list_id = data.get('userListId')
                print(f"    ✓ Successfully submitted. User List ID: {user_list_id}")
                return user_list_id
            else:
                print(f"    ✗ Error submitting gene list: {response.status_code}")
                print(f"    Response text: {response.text[:200]}")
                return None
        except Exception as e:
            print(f"    ✗ Exception submitting gene list: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def get_enrichment_results(self, user_list_id, library_name):
        """Get enrichment results from Enrichr for a specific library."""
        print(f"    Fetching enrichment results for library: {library_name}")
        query_string = f'?userListId={user_list_id}&backgroundType={library_name}'
        full_url = self.enrich_url + query_string
        print(f"    URL: {full_url}")
        
        try:
            response = requests.get(full_url)
            print(f"    Response status code: {response.status_code}")
            
            if response.ok:
                data = response.json()
                results = data.get(library_name, [])
                print(f"    ✓ Retrieved {len(results)} enrichment results")
                if len(results) > 0:
                    print(f"    Top 3 results:")
                    for i in range(min(3, len(results))):
                        print(f"      {i+1}. {results[i][1] if len(results[i]) > 1 else 'N/A'} (p={results[i][2]:.2e})")
                return results
            else:
                print(f"    ✗ Error getting enrichment results: {response.status_code}")
                print(f"    Response text: {response.text[:200]}")
                return []
        except Exception as e:
            print(f"    ✗ Exception getting enrichment results: {e}")
            import traceback
            traceback.print_exc()
            return []
    
    def analyze_enrichment(self):
        """Analyze disease enrichment in each cluster using Enrichr API.

        Every term returned for every cluster is pooled into ONE family and
        corrected once; Enrichr's own per-cluster adjusted p is kept only as
        `enrichr_adjusted_p_value`."""
        alpha = get_alpha()
        print(f"\n{'='*80}")
        print(f"STARTING ENRICHMENT ANALYSIS")
        print(f"{'='*80}")
        
        if not self.clusters:
            print("ERROR: No clusters loaded!")
            return pd.DataFrame(), pd.DataFrame()
        
        print(f"Total clusters to process: {len(self.clusters)}")
        
        # PASS 1: query Enrichr; keep EVERY term's p-value for the family
        pending = {}
        p_chunks = []
        offset = 0
        n_failed = 0
        
        for cluster_idx, (cluster_name, cluster_genes) in enumerate(self.clusters.items()):
            print(f"\n{'-'*80}")
            print(f"Processing cluster {cluster_idx + 1}/{len(self.clusters)}: {cluster_name}")
            print(f"  Cluster size: {len(cluster_genes)} genes")
            
            cluster_genes_list = [str(g) for g in cluster_genes]
            cluster_seed_genes = self.cluster_seed_mapping[cluster_name]
            print(f"  Seed genes: {len(cluster_seed_genes)}")
            
            user_list_id = self.submit_gene_list_to_enrichr(
                cluster_genes_list, 
                description=f"{self.ppi_source}_{cluster_name}"
            )
            
            if not user_list_id:
                print(f"  ✗ Failed to submit gene list for cluster {cluster_name}")
                n_failed += 1
                continue
            
            time.sleep(0.5)
            
            enrichment_results = self.get_enrichment_results(user_list_id, self.library_name)
            
            if not enrichment_results:
                print(f"  ✗ No enrichment results for cluster {cluster_name}")
                n_failed += 1
                continue
            
            print(f"  Processing {len(enrichment_results)} enrichment results...")
            
            cluster_p = [float(r[2]) for r in enrichment_results if len(r) > 2]
            top_offset = offset
            p_chunks.append(cluster_p)
            offset += len(cluster_p)
            
            # Enrichr row: [Rank, Term, P-value, Z-score, Combined Score, Overlapping Genes, Adjusted P-value, ...]
            best_result = enrichment_results[0]
            
            if len(best_result) >= 7:
                term = best_result[1]
                p_value = best_result[2]
                enrichr_adjusted_p = best_result[6]
                overlapping_genes = best_result[5]
                combined_score = best_result[4]
                
                if isinstance(overlapping_genes, str):
                    genes_in_term = overlapping_genes.split(';') if overlapping_genes else []
                    overlapping_genes_str = overlapping_genes
                elif isinstance(overlapping_genes, list):
                    genes_in_term = overlapping_genes
                    overlapping_genes_str = ';'.join(overlapping_genes)
                else:
                    genes_in_term = []
                    overlapping_genes_str = ""
                
                if len(genes_in_term) > 0:
                    pending[cluster_name] = {
                        'cluster_genes': cluster_genes,
                        'cluster_genes_list': cluster_genes_list,
                        'cluster_seed_genes': cluster_seed_genes,
                        'term': term,
                        'p_value': p_value,
                        'enrichr_adjusted_p_value': enrichr_adjusted_p,
                        'combined_score': combined_score,
                        'genes_in_term_count': len(genes_in_term),
                        'overlapping_genes_str': overlapping_genes_str,
                        'rank': best_result[0],
                        'top_offset': top_offset,
                    }
        
        # PASS 2: one correction over every (cluster, term) test
        all_p = np.array([p for chunk in p_chunks for p in chunk], dtype=float)
        all_adj = adjust_pvalues(all_p)
        print(f"\n{'='*80}")
        print(f"MULTIPLE TESTING CORRECTION")
        print(f"{'='*80}")
        print(f"Method: {method_label()}; family = {len(all_p)} tests "
              f"({len(p_chunks)} clusters queried successfully)")
        if n_failed:
            print(f"  ⚠ {n_failed} cluster(s) failed/returned nothing and are NOT in the family; "
                  f"rerun to get a complete family if this is unexpected.")
        
        # PASS 3: build results from each cluster's top term
        summarized_results = []
        gene_level_results = []
        
        for cluster_name, info in pending.items():
            adjusted_p_value = float(all_adj[info['top_offset']])
            percentage = (info['genes_in_term_count'] / len(info['cluster_genes_list'])) * 100
            
            result_dict = create_standardized_results(
                cluster_name=cluster_name,
                cluster_genes=info['cluster_genes'],
                cluster_seed_genes=info['cluster_seed_genes'],
                cluster_type=self.cluster_type,
                total_genes_analyzed=info['genes_in_term_count'],
                percentage=percentage,
                mean_statistic=info['combined_score'],
                p_value=info['p_value'],
                adjusted_p_value=adjusted_p_value,
                statistic_name='combined_score',
                genes_in_expression_data=info['genes_in_term_count']
            )
            result_dict['disease'] = info['term']
            result_dict['genes_in_disease'] = info['genes_in_term_count']
            result_dict['overlapping_genes'] = info['overlapping_genes_str']
            result_dict['rank'] = info['rank']
            result_dict['enrichr_adjusted_p_value'] = info['enrichr_adjusted_p_value']
            summarized_results.append(result_dict)
            
            print(f"  ✓ {cluster_name}: top enrichment {info['term']} "
                  f"(score={info['combined_score']:.2f}, p={info['p_value']:.2e}, adj p={adjusted_p_value:.2e})")
            
            for gene in info['cluster_genes_list']:
                gene_level_results.append({
                    'cluster_id': cluster_name,
                    'gene': gene,
                    'is_seed_gene': gene in info['cluster_seed_genes'],
                    'is_in_expression_data': True,
                    'cluster_combined_score': info['combined_score'],
                    'cluster_p_value': info['p_value'],
                    'cluster_adjusted_p_value': adjusted_p_value,
                    'cluster_top_disease': info['term']
                })
        
        print(f"\n{'='*80}")
        print(f"ENRICHMENT ANALYSIS COMPLETE")
        print(f"{'='*80}")
        print(f"Total disease enrichments across all clusters: {len(summarized_results)}")
        print(f"Total gene-level results: {len(gene_level_results)}")
        
        results_df = pd.DataFrame(summarized_results)
        gene_results_df = pd.DataFrame(gene_level_results)
        
        print(f"\nResults DataFrame shape: {results_df.shape}")
        print(f"Gene Results DataFrame shape: {gene_results_df.shape}")
        
        if not results_df.empty:
            sig_count = (results_df['adjusted_p_value'] < alpha).sum()
            print(f"  Significant top terms (adj p < {alpha}): {sig_count}/{len(results_df)}")
            
            diseases_per_cluster = results_df.groupby('cluster_id').size()
            print(f"\nDiseases per cluster statistics:")
            print(f"  Mean: {diseases_per_cluster.mean():.1f}")
            print(f"  Median: {diseases_per_cluster.median():.1f}")
            print(f"  Max: {diseases_per_cluster.max()}")
            print(f"  Min: {diseases_per_cluster.min()}")
        
        if not gene_results_df.empty:
            print(f"\nAnnotating gene results with cluster significance...")
            gene_results_df['is_cluster_significant'] = gene_results_df['cluster_adjusted_p_value'] < alpha
            sig_count = gene_results_df['is_cluster_significant'].sum()
            print(f"  {sig_count} genes in clusters with significant disease enrichments (adj p < {alpha})")
        
        self.results = results_df
        self.gene_results = gene_results_df
        
        print(f"\n{'='*80}\n")
        return results_df, gene_results_df
    
    def create_visualizations(self):
        """Create standardized visualizations."""
        alpha = get_alpha()
        print(f"\n{'='*80}")
        print(f"CREATING VISUALIZATIONS")
        print(f"{'='*80}")
        
        plots_dir = os.path.join('plots', self.ppi_source, f'orphanet_enrichr_analysis_{self.cluster_type}_{self.safe_prefix}')
        ensure_dir(plots_dir)
        print(f"Plots directory: {plots_dir}")
        
        if self.results.empty:
            print("✗ No results to visualize")
            return
        
        print(f"Creating visualizations for {len(self.results)} results...")
        
        print("\n1. Creating volcano plot...")
        plt.figure(figsize=(10, 8))
        
        self.results['neg_log_pvalue'] = -np.log10(self.results['p_value'] + 1e-300)
        self.results['log2_combined_score'] = self.results['mean_combined_score'].apply(self.safe_log2)
        
        colors = ['red' if pval < alpha else 'gray' for pval in self.results['adjusted_p_value']]
        sig_count = sum(1 for c in colors if c == 'red')
        print(f"   Significant results (red, adj p < {alpha}): {sig_count}/{len(colors)}")
        
        plt.scatter(self.results['log2_combined_score'], self.results['neg_log_pvalue'], 
                   c=colors, alpha=0.6, s=100)
        
        plt.axhline(y=-np.log10(alpha), color='black', linestyle='--', alpha=0.5)
        
        plt.xlabel('log2(Combined Score)')
        plt.ylabel('-log10(p-value)')
        plt.title(f'Orphanet Enrichr Analysis - {self.ppi_source}\n(colored by {method_label()} adj p < {alpha})')
        plt.tight_layout()
        
        volcano_path = os.path.join(plots_dir, 'orphanet_enrichr_volcano_plot.png')
        plt.savefig(volcano_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✓ Saved: {volcano_path}")
        
        print("\n2. Creating top enrichments bar plot...")
        if not self.results.empty:
            top_results = self.results.nlargest(20, 'mean_combined_score')
            print(f"   Plotting top {len(top_results)} enrichments")
            
            plt.figure(figsize=(14, 10))
            y_labels = [f"{row['cluster_id'][:30]}: {row['disease'][:50]}" 
                       for _, row in top_results.iterrows()]
            plt.barh(range(len(top_results)), top_results['mean_combined_score'])
            plt.yticks(range(len(top_results)), y_labels, fontsize=8)
            plt.xlabel('Combined Score')
            plt.title(f'Top 20 Orphanet Enrichments - {self.ppi_source}')
            plt.tight_layout()
            
            bar_path = os.path.join(plots_dir, 'top_enrichments.png')
            plt.savefig(bar_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"   ✓ Saved: {bar_path}")
        
        print(f"\n{'='*80}\n")
    
    def generate_report(self):
        """Generate a simple text report."""
        alpha = get_alpha()
        print(f"\n{'='*80}")
        print(f"GENERATING REPORT")
        print(f"{'='*80}")
        
        report_file = os.path.join(self.analysis_dir, 'analysis_report.txt')
        print(f"Report file: {report_file}")
        
        with open(report_file, 'w') as f:
            f.write(f"Orphanet Enrichr Analysis Report\n")
            f.write(f"=================================\n\n")
            f.write(f"PPI Source: {self.ppi_source}\n")
            f.write(f"Cluster Type: {self.cluster_type}\n")
            f.write(f"Annotation: {self.safe_prefix}\n")
            f.write(f"Enrichr Library: {self.library_name}\n")
            f.write(f"Multiple testing: {method_label()}, alpha={alpha} "
                    f"(pooled over all clusters x library terms)\n")
            f.write(f"Total clusters: {len(self.clusters)}\n\n")
            
            if not self.results.empty:
                f.write(f"Analysis Results:\n")
                f.write(f"Total disease enrichments: {len(self.results)}\n")
                f.write(f"Significant enrichments (adj p < {alpha}): {(self.results['adjusted_p_value'] < alpha).sum()}\n")
                
                diseases_per_cluster = self.results.groupby('cluster_id').size()
                f.write(f"\nDiseases per cluster:\n")
                f.write(f"  Mean: {diseases_per_cluster.mean():.1f}\n")
                f.write(f"  Median: {diseases_per_cluster.median():.1f}\n")
                f.write(f"  Range: {diseases_per_cluster.min()}-{diseases_per_cluster.max()}\n\n")
                
                f.write("Top 20 Enrichments:\n")
                top_results = self.results.nlargest(20, 'mean_combined_score')
                for idx, row in top_results.iterrows():
                    f.write(f"\n  {row['cluster_id']}: {row['disease']}\n")
                    f.write(f"    Combined Score={row['mean_combined_score']:.2f}, ")
                    f.write(f"p={row['p_value']:.2e}, adj_p={row['adjusted_p_value']:.2e}\n")
        
        print(f"✓ Report saved: {report_file}")
        print(f"\n{'='*80}\n")
    
    def run_analysis(self):
        """Run the complete analysis pipeline."""
        print(f"\n{'#'*80}")
        print(f"# STARTING ORPHANET ENRICHR ANALYSIS")
        print(f"# PPI Source: {self.ppi_source}")
        print(f"# Cluster Type: {self.cluster_type}")
        print(f"{'#'*80}\n")
        
        print(f"STEP 1: Loading HiDeF clusters from comparison results...")
        if not self.load_hidef_clusters_multiplex():
            print("✗ Failed to load HiDeF clusters")
            return False

        print(f"✓ Loaded {len(self.clusters)} clusters with {len(self.seed_genes)} seed genes")
        print(f"  Cluster names: {list(self.clusters.keys())[:5]}{'...' if len(self.clusters) > 5 else ''}")
           
        print(f"\nSTEP 2: Running enrichment analysis...")
        results_df, gene_results_df = self.analyze_enrichment()
        
        if not results_df.empty:
            print(f"\nSTEP 3: Saving results...")
            save_results(results_df, gene_results_df, self.analysis_dir, 'orphanet_enrichr')
            print(f"✓ Results saved to: {self.analysis_dir}")
            
            print(f"\nSTEP 4: Creating visualizations...")
            self.create_visualizations()
            
            print(f"\nSTEP 5: Generating report...")
            self.generate_report()
            
            print(f"\n{'#'*80}")
            print(f"# ANALYSIS COMPLETE!")
            print(f"# Results saved to: {self.analysis_dir}")
            print(f"{'#'*80}\n")
            
            return True
        else:
            print("\n✗ No results generated")
            print(f"  - Clusters loaded: {len(self.clusters) if self.clusters else 0}")
            print(f"  - Results DataFrame empty: {results_df.empty}")
            print(f"  - Gene Results DataFrame empty: {gene_results_df.empty}")
            return False

In [3]:
ppi_source='combined26_multiplex'
input_dir='./output'
output_dir='./output'
persistence_threshold=None

analyzer = OrphanetDiseaseEnrichrAnalyzer(
    ppi_source=ppi_source,
    input_dir=input_dir,
    output_dir=output_dir,
    persistence_threshold=persistence_threshold
)
analyzer.run_analysis()


INITIALIZING OrphanetDiseaseEnrichrAnalyzer
PPI Source: combined26_multiplex
Input Dir: ./output
Output Dir: ./output
Multiple testing: Benjamini-Hochberg FDR (alpha=0.05)

Enrichr Configuration:
  Base URL: https://maayanlab.cloud/Enrichr
  Library: Orphanet_Augmented_2021

Output Configuration:
  Analysis directory: ./output/combined26_multiplex/orphanet_enrichr_analysis_hidef_from_comparison
  Directory created/verified: True


################################################################################
# STARTING ORPHANET ENRICHR ANALYSIS
# PPI Source: combined26_multiplex
# Cluster Type: hidef_from_comparison
################################################################################

STEP 1: Loading HiDeF clusters from comparison results...
load_multiplex_clusters(/lisc/home/user/gerardursin/multiplex/11_hidef_module/results_cpm/heterogeneous_hidef_faithful_cpm.nodes): kept 2383 clusters (persistence >= 8); skipped 857 below threshold, 0 empty, 0 below min_size=2
Loaded

Traceback (most recent call last):
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
  File "/lisc/opt/sw/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/socket.py", line 955, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
socket.gaierror: [Errno -2] Name or service not known

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/

    ✗ Exception submitting gene list: HTTPSConnectionPool(host='maayanlab.cloud', port=443): Max retries exceeded with url: /Enrichr/addList (Caused by NameResolutionError("HTTPSConnection(host='maayanlab.cloud', port=443): Failed to resolve 'maayanlab.cloud' ([Errno -2] Name or service not known)"))
  ✗ Failed to submit gene list for cluster HiDeF_Cluster_Cluster1-42

--------------------------------------------------------------------------------
Processing cluster 118/2383: HiDeF_Cluster_Cluster3-34
  Cluster size: 36 genes
  Seed genes: 0
    Submitting 36 genes to Enrichr...
    First 5 genes: ['ATXN1', 'BEX2', 'CCDC146', 'CCDC85B', 'CDR2']
    ✗ Exception submitting gene list: HTTPSConnectionPool(host='maayanlab.cloud', port=443): Max retries exceeded with url: /Enrichr/addList (Caused by NameResolutionError("HTTPSConnection(host='maayanlab.cloud', port=443): Failed to resolve 'maayanlab.cloud' ([Errno -2] Name or service not known)"))
  ✗ Failed to submit gene list for cluster H

Traceback (most recent call last):
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
  File "/lisc/opt/sw/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/socket.py", line 955, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
socket.gaierror: [Errno -2] Name or service not known

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
  File "/lisc/data/scratch/menche/ines/hidef_env/lib/python3.10/site-packages/

    ✗ Exception submitting gene list: HTTPSConnectionPool(host='maayanlab.cloud', port=443): Max retries exceeded with url: /Enrichr/addList (Caused by NameResolutionError("HTTPSConnection(host='maayanlab.cloud', port=443): Failed to resolve 'maayanlab.cloud' ([Errno -2] Name or service not known)"))
  ✗ Failed to submit gene list for cluster HiDeF_Cluster_Cluster3-592

--------------------------------------------------------------------------------
Processing cluster 1703/2383: HiDeF_Cluster_Cluster2-727
  Cluster size: 4 genes
  Seed genes: 0
    Submitting 4 genes to Enrichr...
    First 5 genes: ['ATF4', 'NAP1L2', 'NAP1L5', 'SNRPB2']
    ✗ Exception submitting gene list: HTTPSConnectionPool(host='maayanlab.cloud', port=443): Max retries exceeded with url: /Enrichr/addList (Caused by NameResolutionError("HTTPSConnection(host='maayanlab.cloud', port=443): Failed to resolve 'maayanlab.cloud' ([Errno -2] Name or service not known)"))
  ✗ Failed to submit gene list for cluster HiDeF_Clu

True

In [4]:
pwd

'/lisc/home/user/gerardursin/multiplex/phen'